# GRU + HGT — Training & Evaluation

Charles's model. HGT encodes the heterogeneous urban graph → region embeddings → GRU encodes the prefix sequence → MLP predicts destination.

Run `build_centroids.py` before this notebook if `gru_param_config.json`, `cell_centroids.pt`, and `taxi_id_map.pt` don't exist yet.

In [1]:
import json
import datetime
import time
from pathlib import Path
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

from hgt_model import HGTDestinationModel
from eval import evaluate_model, print_results_table

DATA_DIR = Path('porto_data_bundle')
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

d:\Anaconda\envs\cpsc583\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 1. Load configs and lookup tables

In [2]:
with open(DATA_DIR / 'gru_param_config.json') as f:
    cfg = json.load(f)
print('GRU config:', cfg)

taxi_id_map = torch.load(DATA_DIR / 'taxi_id_map.pt')
centroids   = torch.load(DATA_DIR / 'cell_centroids.pt')
print(f'Taxi IDs: {len(taxi_id_map)}  |  Centroids: {len(centroids)}')

GRU config: {'num_regions': 6750, 'num_dest_classes': 4978, 'num_taxi_ids': 438}
Taxi IDs: 438  |  Centroids: 6750


## 2. Load heterogeneous graph onto GPU

In [3]:
# Load once at startup — stays on GPU for the entire training run.
# The HGT encoder takes this as input every forward pass.
graph = torch.load(DATA_DIR / 'hetero_graph.pt', map_location=DEVICE, weights_only=False)
print(graph)

HeteroData(
  region={
    num_nodes=6750,
    x=[6750, 150],
  },
  poi={
    num_nodes=2708,
    x=[2708, 144],
  },
  road={
    num_nodes=9445,
    x=[9445, 1],
  },
  (region, taxi_transition, region)={
    edge_index=[2, 59514],
    edge_weight=[59514],
  },
  (region, rev_taxi_transition, region)={
    edge_index=[2, 59514],
    edge_weight=[59514],
  },
  (poi, located_in, region)={ edge_index=[2, 2708] },
  (region, has_poi, poi)={ edge_index=[2, 2708] },
  (road, intersects, region)={ edge_index=[2, 14064] },
  (region, has_road, road)={ edge_index=[2, 14064] },
  (road, connects_to, road)={ edge_index=[2, 23251] }
)


## 3. Dataset and collate_fn

In [ ]:
class ShardDataset(Dataset):
    """Loads all shards for a split into a flat list of example dicts."""
    def __init__(self, shard_paths, desc='Loading'):
        self.examples = []
        for p in tqdm(shard_paths, desc=desc, unit='shard'):
            self.examples.extend(
                torch.load(p, map_location='cpu', weights_only=False)
            )

    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


CALL_TYPE_MAP = {'A': 0, 'B': 1, 'C': 2}
DAY_TYPE_MAP  = {'A': 0, 'B': 1, 'C': 2}


def collate_fn(batch, taxi_id_map):
    seqs    = [torch.tensor(ex['prefix_region_seq'], dtype=torch.long) for ex in batch]
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    prefix_ids = pad_sequence(seqs, batch_first=True, padding_value=0)

    dest_region = torch.tensor([ex['dest_region'] for ex in batch], dtype=torch.long)
    dest_lat    = torch.tensor([ex['dest_lat']    for ex in batch], dtype=torch.float)
    dest_lon    = torch.tensor([ex['dest_lon']    for ex in batch], dtype=torch.float)

    hours, dows = [], []
    for ex in batch:
        dt = datetime.datetime.fromtimestamp(ex['timestamp'], datetime.timezone.utc)
        hours.append(dt.hour)
        dows.append(dt.weekday())

    return {
        'prefix_ids' : prefix_ids,
        'lengths'    : lengths,
        'dest_region': dest_region,
        'dest_lat'   : dest_lat,
        'dest_lon'   : dest_lon,
        'metadata': {
            'call_type': torch.tensor([CALL_TYPE_MAP[ex['call_type']] for ex in batch], dtype=torch.long),
            'taxi_id'  : torch.tensor([taxi_id_map.get(ex['taxi_id'], 0) for ex in batch], dtype=torch.long),
            'day_type' : torch.tensor([DAY_TYPE_MAP[ex['day_type']]   for ex in batch], dtype=torch.long),
            'hour'     : torch.tensor(hours, dtype=torch.long),
            'dow'      : torch.tensor(dows,  dtype=torch.long),
        }
    }

## 4. Build DataLoaders

Loading all train shards takes a few minutes — they are kept in RAM for the entire run.

In [5]:
BATCH_SIZE = 256
_collate   = partial(collate_fn, taxi_id_map=taxi_id_map)

train_paths = sorted((DATA_DIR / 'supervised_shards' / 'train').glob('train_*.pt'))
val_paths   = sorted((DATA_DIR / 'supervised_shards' / 'val').glob('val_*.pt'))
test_paths  = sorted((DATA_DIR / 'supervised_shards' / 'test').glob('test_*.pt'))

print('Loading val and test shards...')
val_dataset  = ShardDataset(val_paths)
test_dataset = ShardDataset(test_paths)

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=_collate, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=_collate, num_workers=2, pin_memory=True)

print(f'Val: {len(val_dataset):,}  |  Test: {len(test_dataset):,}')

# Train shards loaded inside the seed loop to allow garbage collection between seeds
print('Val/Test loaders ready. Train data loaded per seed.')

Loading val and test shards...


Loading: 100%|██████████| 11/11 [00:10<00:00,  1.06shard/s]

Val: 1,062,639  |  Test: 1,061,430
Val/Test loaders ready. Train data loaded per seed.


## 5. Training loop (single seed)

In [ ]:
def train_seed(seed: int) -> dict:
    """
    Train HGTDestinationModel with a fixed random seed.
    Both HGT and GRU use true SGD — both optimizers step every batch.
    Returns test set results for the best checkpoint (by val Recall@5).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f'\n{"="*60}')
    print(f'SEED {seed}')
    print(f'{"="*60}')

    in_channels_dict = {ntype: graph[ntype].x.shape[1] for ntype in graph.node_types}

    model = HGTDestinationModel(
        metadata         = graph.metadata(),
        in_channels_dict = in_channels_dict,
        num_regions      = cfg['num_regions'],
        num_dest_classes = cfg['num_dest_classes'],
        num_taxi_ids     = cfg['num_taxi_ids'],
        hidden_dim       = 64,
        num_heads        = 4,
        num_layers       = 2,
        gru_hidden       = 128,
        gru_layers       = 2,
        dropout          = 0.2,
    ).to(DEVICE)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model parameters: {num_params:,}')

    # Two optimizers — both step every batch (true SGD for both HGT and GRU)
    gru_optimizer = torch.optim.Adam(model.gru.parameters(), lr=1e-3)
    hgt_optimizer = torch.optim.Adam(model.hgt.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    ckpt_path = f'hgt_best_seed{seed}.pt'

    train_dataset = ShardDataset(train_paths, desc=f'Loading train shards (seed {seed})')
    train_loader  = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=_collate, num_workers=2, pin_memory=True
    )
    print(f'Train: {len(train_dataset):,} examples  |  {len(train_loader):,} batches/epoch\n')

    best_recall5     = 0.0
    patience_counter = 0
    PATIENCE         = 5
    MAX_EPOCHS       = 30
    n_batches        = len(train_loader)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss  = 0.0
        epoch_start = time.time()

        pbar = tqdm(train_loader, total=n_batches,
                    desc=f'Epoch {epoch:02d}/{MAX_EPOCHS}', unit='batch', leave=True)
        for batch in pbar:
            prefix_ids = batch['prefix_ids'].to(DEVICE)
            lengths    = batch['lengths']
            dest       = batch['dest_region'].to(DEVICE)
            meta       = {k: v.to(DEVICE) for k, v in batch['metadata'].items()}

            # HGT runs every batch — true SGD for the graph encoder
            region_embs = model.hgt(graph)

            logits = model(prefix_ids, lengths, meta, region_embs=region_embs)
            loss   = criterion(logits, dest)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            gru_optimizer.step();  gru_optimizer.zero_grad()
            hgt_optimizer.step();  hgt_optimizer.zero_grad()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss   = total_loss / n_batches
        epoch_time = time.time() - epoch_start

        print(f'  Epoch {epoch:02d} | train loss {avg_loss:.4f} | '
              f'time {epoch_time/60:.1f} min | validating...', end=' ', flush=True)

        val_results = evaluate_model(model, val_loader, centroids, DEVICE,
                                     graph_data=graph)
        recall5 = val_results['Recall@5']

        print(f'val R@1 {val_results["Recall@1"]:.4f} | '
              f'val R@5 {recall5:.4f} | '
              f'val R@10 {val_results["Recall@10"]:.4f} | '
              f'mean H {val_results["Mean Haversine (km)"]:.3f} km', end='')

        if recall5 > best_recall5:
            best_recall5 = recall5
            torch.save(model.state_dict(), ckpt_path)
            patience_counter = 0
            print('  ← best')
        else:
            patience_counter += 1
            print(f'  (patience {patience_counter}/{PATIENCE})')
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}.')
                break

    print(f'\nLoading best checkpoint (val R@5 = {best_recall5:.4f}) and evaluating on test...')
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
    test_results = evaluate_model(model, test_loader, centroids, DEVICE,
                                  graph_data=graph)
    print(f'[Seed {seed}] Test R@1 {test_results["Recall@1"]:.4f} | '
          f'R@5 {test_results["Recall@5"]:.4f} | '
          f'R@10 {test_results["Recall@10"]:.4f} | '
          f'Mean H {test_results["Mean Haversine (km)"]:.3f} km | '
          f'Med H {test_results["Med Haversine (km)"]:.3f} km')
    return test_results


## 6. Run 3 seeds — report mean ± std

In [ ]:
SEEDS = [0, 1, 2]
seed_results = []

for seed in SEEDS:
    result = train_seed(seed)
    seed_results.append(result)

metrics = ['Recall@1', 'Recall@5', 'Recall@10', 'Mean Haversine (km)', 'Med Haversine (km)']
print('\n=== GRU + HGT — 3-seed summary ===')
for m in metrics:
    vals = [r[m] for r in seed_results]
    print(f'  {m:<25}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

## 7. Final results table (fill in other models as they complete)

In [ ]:
hgt_avg = {m: float(np.mean([r[m] for r in seed_results])) for m in metrics}
hgt_avg['n'] = seed_results[0]['n']

print_results_table({
    'GRU + HGT (ours)': hgt_avg,
    # 'Markov Baseline' : markov_results,   # paste from markov.ipynb
    # 'GRU + R-GCN'     : rgcn_results,     # paste from Yidan's notebook
})